<a href="https://colab.research.google.com/github/PradeepK-eng/Google-Colab-Files/blob/main/MeatQuality_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥩 Meat Quality AI — Colab Notebook
Generated by MeatScan AI v2.0

# 🥩 Meat Quality AI — Google Colab Pipeline
**Scan ID:** SC-MMSVIB2D · **Date:** Mon Mar 16 2026 · **Meat:** Beef

Complete step-by-step ML pipeline from ESP32 sensor data collection to freshness prediction and deployment.

#Stage 1 — Environment Setup
Install all required libraries for data processing, ML model training, CNN image analysis, and deployment.

In [ ]:
# Stage 1: Install all required packages
!pip install pandas numpy scikit-learn opencv-python matplotlib seaborn
!pip install tensorflow joblib imbalanced-learn
!pip install fastapi uvicorn pyngrok streamlit
print("✓ All packages installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 110.2 MB/s eta 0:00:00
✓ All packages installed successfully


#Stage 2 — Create / Load Dataset
Build the sensor dataset with labeled freshness classes. Columns: gas (ppm), ph, temperature (°C), humidity (%), storage_days, label.

In [ ]:
import pandas as pd
import numpy as np

# Use your own: df = pd.read_csv('meat_sensor_data.csv')
data = {
  'gas':         [210,320,500,180,450,280,150,390,550,120],
  'ph':          [5.6,6.3,6.9,5.5,6.7,6.1,5.4,6.5,7.0,5.3],
  'temperature': [4,5,7,3,8,5,4,6,9,3],
  'humidity':    [60,68,75,55,78,65,58,72,80,52],
  'storage_days':[3,5,8,2,7,4,2,6,10,1],
  'label':       ['Fresh','Consume','Spoiled','Fresh','Spoiled','Consume','Fresh','Consume','Spoiled','Fresh']
}
df = pd.DataFrame(data)
print(df)
print(f"\nClass distribution:\n{df['label'].value_counts()}")

   gas   ph  temperature  humidity  storage_days    label
0  210  5.6            4        60             3    Fresh
1  320  6.3            5        68             5  Consume
2  500  6.9            7        75             8  Spoiled
3  180  5.5            3        55             2    Fresh
4  450  6.7            8        78             7  Spoiled
5  280  6.1            5        65             4  Consume
6  150  5.4            4        58             2    Fresh
7  390  6.5            6        72             6  Consume
8  550  7.0            9        80            10  Spoiled
9  120  5.3            3        52             1    Fresh

Class distribution:
label
Fresh      4
Consume    3
Spoiled    3
Name: count, dtype: int64


#Stage 3 — Preprocessing & Feature Engineering
Standardize features, encode labels, handle class imbalance with SMOTE, perform 80/20 train-test split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

X = df[['gas','ph','temperature','humidity','storage_days']]
y = df['label']
le = LabelEncoder()
y_enc = le.fit_transform(y)

# SMOTE: synthetic minority oversampling
sm = SMOTE(random_state=42)
X_bal, y_bal = sm.fit_resample(X, y_enc)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_bal)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_bal, test_size=0.2, random_state=42, stratify=y_bal)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

ValueError: Expected n_neighbors <= n_samples_fit, but n_neighbors = 6, n_samples_fit = 3, n_samples = 3

#Stage 4 — Train Random Forest (GridSearchCV)
Hyperparameter tuning with 5-fold cross-validation. Evaluate accuracy, precision, recall, F1-score.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

param_grid = {'n_estimators':[100,200], 'max_depth':[None,10], 'min_samples_split':[2,5]}
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
gs = GridSearchCV(rf, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)
best = gs.best_estimator_
pred = best.predict(X_test)
print("Best params:", gs.best_params_)
print(f"Accuracy: {accuracy_score(y_test,pred):.4f}")
print(classification_report(y_test, pred, target_names=le.classes_))

#Stage 5 — CNN Image Model (MobileNetV2 Transfer Learning)
Freeze ImageNet base weights, add custom classification head for Fresh / Consume / Spoiled.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# Load MobileNetV2 pre-trained on ImageNet, no top layer
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False  # Freeze base weights

# Custom classification head
x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(3, activation='softmax')(x)  # 3 classes
cnn = Model(base.input, out)
cnn.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
print(f"Total params: {cnn.count_params():,}")
print(f"Trainable: {sum(v.numpy().size for v in cnn.trainable_weights):,}")

In [ ]:
#Stage 6 — Predict Current Sample
Run the live sensor readings from this scan (Scan ID: SC-MMSVIB2D) through both trained models.

In [ ]:
import joblib

# Load saved models
rf_model = joblib.load('models/meat_quality_model.pkl')
scaler   = joblib.load('models/scaler.pkl')
le       = joblib.load('models/label_encoder.pkl')

# Live sensor input — Scan ID: SC-MMSVIB2D
sample = np.array([[210.0, 5.6, 4.0, 60, 3.0]])
s_scaled = scaler.transform(sample)

pred  = rf_model.predict(s_scaled)
proba = rf_model.predict_proba(s_scaled)

print(f"Prediction   : {le.inverse_transform(pred)[0]}")
print(f"Confidence   : {proba.max()*100:.1f}%")
print(f"Freshness Score: 0/100")
print(f"Verdict      : SPOILED")

#Stage 7 — Save & Export for Deployment
Save all trained artifacts for FastAPI + Streamlit deployment.

In [ ]:
import joblib, os
os.makedirs('models', exist_ok=True)

joblib.dump(best,   'models/meat_quality_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(le,     'models/label_encoder.pkl')
cnn.save(         'models/cnn_meat_model.h5')

print("✓ Random Forest → models/meat_quality_model.pkl")
print("✓ Scaler       → models/scaler.pkl")
print("✓ CNN model    → models/cnn_meat_model.h5")
print("→ Run: uvicorn app.main:app --reload")
print("→ Run: streamlit run streamlit_app.py")